# Cell 6 — Ultra Memory-Efficient Pipeline
Uses pure NumPy matrix multiplication instead of xarray broadcasting.
Processes ALL 3663 regions at once using only ~200MB RAM total.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import os, gc
from tqdm.notebook import tqdm

# ── PATHS ──────────────────────────────────────────────────────────────
SIMULATION_DIR   = '/media/wcs/Disk3/imran/GCM_models/CNRM-ESM2-1/RegridedMonthly'
TARGET_DIR       = '/media/wcs/Disk3/imran/GCM_models/CNRM-ESM2-1/regional_averages_output'
MASKS_PATH       = '/media/wcs/Disk3/imran/GCM_models/masks/subregion_masks.nc'
INDICATOR        = 'tas'
SCENARIOS_I_WANT = ['historical','ssp126','ssp245','ssp370','ssp460','ssp585']

print("Paths configured ✅")


## Step A — Pre-load ALL masks as one NumPy matrix (run once)

In [ ]:
# Load ALL 3663 mask variables into a single 2D numpy matrix
# Shape: (n_regions, n_grid_cells) = (3663, 72×144) = (3663, 10368)
# Memory: 3663 × 10368 × 4 bytes (float32) = ~152 MB — fits easily

print("Loading masks into NumPy matrix...")

masks_ds     = xr.open_dataset(MASKS_PATH)
region_names = list(masks_ds.data_vars)
lat_vals     = masks_ds.lat.values
lon_vals     = masks_ds.lon.values

n_regions  = len(region_names)
n_lat      = len(lat_vals)
n_lon      = len(lon_vals)
n_cells    = n_lat * n_lon

# cos(lat) weights — shape (n_lat,) then broadcast to (n_lat, n_lon)
cos_lat_2d = np.cos(np.deg2rad(lat_vals))[:, np.newaxis] * np.ones((1, n_lon))
cos_lat_2d = cos_lat_2d.astype(np.float32)             # shape: (72, 144)
cos_lat_flat = cos_lat_2d.ravel()                       # shape: (10368,)

# Build the full mask matrix in one shot
# Shape: (n_regions, n_cells)
masks_matrix = np.zeros((n_regions, n_cells), dtype=np.float32)

for i, region in enumerate(tqdm(region_names, desc="Building mask matrix")):
    mask_2d = masks_ds[region].values.astype(np.float32)   # (72, 144)
    weighted = mask_2d * cos_lat_2d                         # apply cos(lat)
    weighted[weighted == 0] = np.nan                        # 0 → NaN
    masks_matrix[i] = weighted.ravel()                      # flatten to (10368,)

masks_ds.close()

# Precompute denominators (sum of weights per region) — shape: (n_regions,)
denominators = np.nansum(masks_matrix, axis=1)             # (3663,)

# Replace NaN with 0 in mask matrix for multiplication
# (NaN × data = NaN, but we want 0 contribution from masked cells)
masks_matrix_clean = np.where(np.isnan(masks_matrix), 0.0, masks_matrix)

# Free the NaN version — we only need the clean one now
del masks_matrix
gc.collect()

print(f"\nMask matrix ready!")
print(f"  Shape       : {masks_matrix_clean.shape}")
print(f"  Memory      : {masks_matrix_clean.nbytes / 1e6:.1f} MB")
print(f"  Regions     : {n_regions}")
print(f"  Grid cells  : {n_cells} ({n_lat} lat × {n_lon} lon)")
print(f"  Valid regs  : {(denominators > 0).sum()} (rest are ocean/tiny islands)")


In [ ]:
# Find simulation files
all_files = [
    f for f in os.listdir(SIMULATION_DIR)
    if f.endswith('.nc')
    and len(f.split('_')) > 3
    and f.split('_')[3] in SCENARIOS_I_WANT
]

print(f"Files found: {len(all_files)}")
for f in all_files:
    print(f"  {f}")


## Step B — Define the memory-efficient function

In [ ]:
def process_file_numpy(
    simulation_directory,
    simulation_file,
    target_directory,
    masks_matrix_clean,   # (n_regions, n_cells) float32 numpy
    denominators,         # (n_regions,) float32 numpy
    region_names,         # list of region variable names
    cos_lat_flat,         # (n_cells,) for global mean
    indicator,
    need_global_mean=False,
):
    simulation_path = f"{simulation_directory}/{simulation_file}"

    MODEL     = simulation_file.split('_')[2]
    scenario  = simulation_file.split('_')[3]
    ensemble  = simulation_file.split('_')[4]
    SCENARIO  = f"{scenario}-{ensemble}"
    INDICATOR = simulation_file.split('_')[0]

    print(f"\nProcessing: {MODEL} | {SCENARIO} | {INDICATOR}")
    print(f"  Loading NetCDF...")

    # ── Load simulation — only the indicator variable ─────────────────
    with xr.open_dataset(simulation_path) as ds:
        ds['lon'] = xr.where(ds['lon'] > 180, ds['lon'] - 360, ds['lon'])
        ds        = ds.sortby('lon')
        sim_np    = ds[indicator].values.astype(np.float32)  # (time, lat, lon)
        time_vals = ds['time'].values

    n_time = sim_np.shape[0]
    print(f"  Simulation shape : {sim_np.shape}, dtype={sim_np.dtype}")
    print(f"  Sim memory       : {sim_np.nbytes / 1e6:.1f} MB")

    # ── Reshape simulation: (time, lat×lon) ──────────────────────────
    sim_flat = sim_np.reshape(n_time, -1)   # (time, n_cells)
    del sim_np
    gc.collect()

    # ── Global mean (optional) ────────────────────────────────────────
    if need_global_mean:
        denom_global = cos_lat_flat.sum()
        global_ts    = (sim_flat * cos_lat_flat[np.newaxis, :]).sum(axis=1) / denom_global
        df_global    = pd.DataFrame({'time': time_vals, 'tas': global_ts})
        gm_dir       = f"{target_directory}/cmip-6_global_mean/{indicator}"
        os.makedirs(gm_dir, exist_ok=True)
        df_global.to_csv(
            f"{gm_dir}/globalmean_{INDICATOR.lower()}_{MODEL.lower()}_{SCENARIO.lower()}.csv",
            index=False
        )
        print(f"  Global mean saved ✅")
        del global_ts, df_global

    # ── KEY STEP: matrix multiply to get ALL regions × ALL timesteps ──
    # sim_flat:           (n_time,    n_cells)
    # masks_matrix_clean: (n_regions, n_cells)
    # result:             (n_time,    n_regions)  via sim_flat @ masks.T
    print(f"  Computing all regional averages via matrix multiply...")

    # (n_time, n_cells) @ (n_cells, n_regions) = (n_time, n_regions)
    all_averages = sim_flat @ masks_matrix_clean.T   # shape: (n_time, n_regions)

    # Divide each region column by its denominator
    all_averages /= denominators[np.newaxis, :]      # broadcast divide

    print(f"  Averages matrix shape: {all_averages.shape}")
    print(f"  Averages memory      : {all_averages.nbytes / 1e6:.1f} MB")

    del sim_flat
    gc.collect()

    # ── Write one CSV per region ──────────────────────────────────────
    print(f"  Writing {len(region_names)} CSVs...")
    skipped = 0

    for i, region in enumerate(tqdm(region_names, desc=f"  Writing CSVs")):
        if denominators[i] == 0:
            skipped += 1
            continue   # skip regions with no valid grid cells

        region_clean = region.replace('m_', '')
        ts           = all_averages[:, i]

        df = pd.DataFrame({'time': time_vals, region_clean: ts})

        out_dir  = f"{target_directory}/isimip_regional_data/{region_clean}/latWeight"
        filename = f"{MODEL}_{SCENARIO}_{INDICATOR}_{region_clean}_latweight.csv".lower()
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(f"{out_dir}/{filename}", index=False)

    del all_averages
    gc.collect()

    print(f"  Done ✅  Written: {len(region_names) - skipped}  Skipped: {skipped}")


print("Function defined ✅")


## Step C — Run the pipeline

In [ ]:
need_global_mean = (INDICATOR == 'tas')

print(f"Starting pipeline")
print(f"Files      : {len(all_files)}")
print(f"Regions    : {len(region_names)}")
print(f"Global mean: {need_global_mean}")
print("=" * 55)

for file in all_files:
    process_file_numpy(
        simulation_directory = SIMULATION_DIR,
        simulation_file      = file,
        target_directory     = TARGET_DIR,
        masks_matrix_clean   = masks_matrix_clean,
        denominators         = denominators,
        region_names         = region_names,
        cos_lat_flat         = cos_lat_flat,
        indicator            = INDICATOR,
        need_global_mean     = need_global_mean,
    )

print("=" * 55)
print("PIPELINE COMPLETE ✅")


In [ ]:
# ── VERIFY OUTPUTS ───────────────────────────────────────────────────
reg_dir = f"{TARGET_DIR}/isimip_regional_data"
gm_dir  = f"{TARGET_DIR}/cmip-6_global_mean/{INDICATOR}"

if os.path.exists(gm_dir):
    print("Global mean CSVs:")
    for f in os.listdir(gm_dir):
        df = pd.read_csv(f"{gm_dir}/{f}")
        print(f"  {f}  →  {len(df)} rows, "
              f"{df['time'].iloc[0]} to {df['time'].iloc[-1]}")

if os.path.exists(reg_dir):
    regions_written = os.listdir(reg_dir)
    print(f"\nRegional CSVs written for {len(regions_written)} regions")
    print(f"Sample: {regions_written[:5]}")

    # Preview one region
    r   = regions_written[0]
    lw  = f"{reg_dir}/{r}/latWeight"
    f   = os.listdir(lw)[0]
    df  = pd.read_csv(f"{lw}/{f}")
    print(f"\nPreview — {f}:")
    print(df.head(3).to_string(index=False))
